## 연습 1: AutoSphere 데이터 플랫폼 설정

당신은 **AutoSphere AG**의 데이터 엔지니어입니다. 글로벌 자동차 제조업체입니다. 연결된 차량 플랫폼은 수백만 대의 차량에서 원격 측정을 수집하고, 고객 등록 데이터를 관리하며, 서비스 기록을 추적합니다.

거버넌스 제어를 적용하기 전에 플랫폼의 기초를 형성하는 카탈로그, 스키마 및 테이블을 생성해야 합니다. 또한 **주석**을 추가하여 각 테이블과 열의 목적을 문서화합니다. 이렇게 하면 다른 팀이 데이터 자산을 쉽게 찾을 수 있습니다.

### 설정 — 카탈로그, 스키마 및 테이블 생성

거버넌스 제어에 집중하기 위해 AutoSphere 데이터 플랫폼은 아래 셀에서 설정됩니다. 이는 다음을 생성합니다:

- **`automotive_catalog`** — AutoSphere 플랫폼용 Unity Catalog 카탈로그(설명 주석 포함)
- **`automotive_catalog.governance_lab`** — 이 실습 전체에서 사용되는 스키마
- **`customer_registrations`** — 등록된 고객 1명당 1개 행 및 관련 차량; 테이블 수준 주석 포함
- **`vehicle_telemetry`** — 연결된 차량에서 연속 원격 측정 이벤트; `speed_kmh` 및 `battery_level_pct`에 대한 열 수준 주석 포함

셀을 실행하여 환경을 준비한 후 연습 2로 계속하십시오.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS automotive_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Connected vehicle platform catalog for AutoSphere AG — stores registration, telemetry, and service data.'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS automotive_catalog.governance_lab
  COMMENT 'Governance lab schema for hands-on Unity Catalog exercises.';

-- Create the customer_registrations table
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.customer_registrations
  COMMENT 'One row per registered AutoSphere customer and their associated vehicle.'
AS SELECT * FROM VALUES
  (1, 'Lukas Bauer',    'lukas.bauer@autosphere.de',   'DE-LB-4821', 'DE', 'VH-001'),
  (2, 'Sophie Martin',  'sophie.martin@autosphere.fr',  'FR-SM-9034', 'FR', 'VH-002'),
  (3, 'James Clarke',   'james.clarke@autosphere.uk',   'UK-JC-1172', 'UK', 'VH-003'),
  (4, 'Yuki Tanaka',    'yuki.tanaka@autosphere.jp',    'JP-YT-5561', 'JP', 'VH-004'),
  (5, 'Maria Santos',   'maria.santos@autosphere.pt',   'PT-MS-8843', 'PT', 'VH-005'),
  (6, 'Carlos Rivera',  'carlos.rivera@autosphere.es',  'ES-CR-3309', 'ES', 'VH-006')
AS t(customer_id, full_name, email, driver_license_no, country, vehicle_id);

-- Create the vehicle_telemetry table
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.vehicle_telemetry (
  event_id          BIGINT    COMMENT 'Unique telemetry event identifier',
  vehicle_id        STRING,
  event_time        TIMESTAMP,
  speed_kmh         INT       COMMENT 'Instantaneous vehicle speed in kilometres per hour',
  battery_level_pct INT       COMMENT 'State of charge as a percentage (0–100)',
  latitude          DOUBLE,
  longitude         DOUBLE,
  country           STRING
);

INSERT INTO automotive_catalog.governance_lab.vehicle_telemetry VALUES
  (1, 'VH-001', TIMESTAMP '2026-03-01 08:15:00', 112, 78,  48.8566,   2.3522, 'DE'),
  (2, 'VH-002', TIMESTAMP '2026-03-01 08:16:00', 95,  55,  51.5074,  -0.1278, 'FR'),
  (3, 'VH-003', TIMESTAMP '2026-03-01 08:17:00', 130, 91,  40.4168,  -3.7038, 'UK'),
  (4, 'VH-004', TIMESTAMP '2026-03-01 08:18:00', 88,  34,  35.6762, 139.6503, 'JP'),
  (5, 'VH-005', TIMESTAMP '2026-03-01 08:19:00', 105, 62,  38.7223,  -9.1393, 'PT'),
  (6, 'VH-006', TIMESTAMP '2026-03-01 08:20:00', 77,  47,  41.3851,   2.1734, 'ES');

## 연습 2: 거버넌스를 위해 데이터 자산 태그 지정

AutoSphere의 준수 팀은 **개인식별정보(PII)**를 포함하는 모든 테이블과 열에 일관되게 레이블을 지정하도록 요구합니다. 태그는 자동 검색을 활성화하고 나중에 액세스 정책을 유도합니다.

이 연습에서는 `customer_registrations` 테이블을 분류하기 위해 테이블 수준 및 열 수준 태그를 적용합니다.

### 작업 2.1 — 테이블 수준 태그 추가

`customer_registrations` 테이블을 도메인 및 민감도별로 분류하기 위해 태그합니다:

- `domain` = `customer`
- `data_classification` = `confidential`

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks SQL의 Unity Catalog 테이블에 키-값 태그를 추가하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER TABLE ... SET TAGS ('key' = 'value', 'key2' = 'value2')`를 사용하십시오.

In [ ]:
%sql
ALTER TABLE automotive_catalog.governance_lab.customer_registrations
SET TAGS ('domain' = 'customer', 'data_classification' = 'confidential');

### 작업 2.2 — 열 수준 PII 태그 추가

`customer_registrations`의 PII 열에 태그를 지정하여 민감한 것으로 표시합니다:

- 열 `email` → `pii` = `email`
- 열 `driver_license_no` → `pii` = `driver_license`

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks의 SQL을 사용하여 Unity Catalog 테이블의 특정 열에 태그를 추가하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER TABLE ... ALTER COLUMN <col> SET TAGS (...)`를 사용하십시오.

In [ ]:
%sql
ALTER TABLE automotive_catalog.governance_lab.customer_registrations
ALTER COLUMN email
SET TAGS ('pii' = 'email');

ALTER TABLE automotive_catalog.governance_lab.customer_registrations
ALTER COLUMN driver_license_no
SET TAGS ('pii' = 'driver_license');

### 작업 2.3 — 태그 확인

`system.information_schema.table_tags`를 사용하여 테이블 수준 태그가 적용되었는지 확인합니다. 그런 다음 열을 설명하여 열 태그가 있는지 확인합니다. 이를 위해 `system.information_schema.column_tags`를 사용하십시오.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks SQL에서 Unity Catalog 테이블 및 해당 열의 태그를 확인하려면 어떻게 해야 합니까?"*

In [ ]:
%sql
-- Show table-level tags for customer_registrations

SELECT
  catalog_name,
  schema_name,
  table_name,
  tag_name,
  tag_value
FROM system.information_schema.table_tags
WHERE catalog_name = 'automotive_catalog'
  AND schema_name = 'governance_lab'
  AND table_name = 'customer_registrations';

In [ ]:
%sql
-- Show column-level tags for customer_registrations

SELECT
  catalog_name,
  schema_name,
  table_name,
  column_name,
  tag_name,
  tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'automotive_catalog'
  AND schema_name = 'governance_lab'
  AND table_name = 'customer_registrations'
ORDER BY column_name, tag_name;

## 연습 3: 데이터 보존 구성

AutoSphere의 차량 원격 측정 테이블은 계속 기록됩니다. GDPR 데이터 최소화 원칙을 준수하고 저장소 비용을 제어하려면 플랫폼은 다음을 수행해야 합니다:

- **14일**의 테이블 히스토리만 유지합니다.
- `VACUUM`을 사용하여 오래된 데이터 파일을 영구적으로 제거합니다.
- **예측 최적화**를 활성화하여 향후 유지보수 실행이 자동으로 수행되도록 합니다.

이 연습에서는 보존 설정을 구성하고, 레코드 삭제를 시뮬레이션하며, `VACUUM` 작업을 실행합니다.

### 작업 3.1 — 보존 속성 구성

`vehicle_telemetry`에서 다음 Delta Lake 테이블 속성을 설정합니다:

- `delta.logRetentionDuration` = `interval 14 days`
- `delta.deletedFileRetentionDuration` = `interval 14 days`

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks의 Unity Catalog 테이블에서 Delta Lake 보존 기간 속성을 설정하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER TABLE ... SET TBLPROPERTIES (...)`를 사용하십시오.

In [ ]:
%sql
ALTER TABLE automotive_catalog.governance_lab.vehicle_telemetry
SET TBLPROPERTIES (
  'delta.logRetentionDuration'         = 'interval 14 days',
  'delta.deletedFileRetentionDuration' = 'interval 14 days'
);

### 작업 3.2 — GDPR 삭제 요청 시뮬레이션

고객이 삭제 권리 요청을 제출했습니다. `vehicle_telemetry` 테이블에서 차량 `VH-003`의 모든 원격 측정 레코드를 삭제합니다.

삭제 후 테이블을 쿼리하여 레코드가 삭제되었는지 확인합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks SQL에서 Delta Lake 테이블의 특정 행을 삭제하려면 어떻게 해야 합니까?"*

In [ ]:
%sql
DELETE FROM automotive_catalog.governance_lab.vehicle_telemetry
WHERE vehicle_id = 'VH-003';

-- Verify the deletion
SELECT * FROM automotive_catalog.governance_lab.vehicle_telemetry
ORDER BY event_id;

### 작업 3.3 — 삭제된 데이터 파일을 제거하기 위해 VACUUM 실행

`VACUUM`이 실행될 때까지 삭제된 행은 기본 Parquet 파일에 남아 있습니다. `vehicle_telemetry`에서 `VACUUM`을 실행하여 더 이상 현재 또는 최근 테이블 버전에서 참조하지 않는 파일을 영구적으로 제거합니다.

**168시간(7일)**의 기본 보존을 사용합니다. 이는 안전하며, 서버리스 컴퓨팅에서 작동하며, 실습 환경에 적합합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks에서 Delta Lake 테이블에 VACUUM을 실행하려면 어떻게 해야 합니까? RETAIN 옵션은 무엇을 합니까?"*

**참고:** 서버리스 컴퓨팅에서는 최소 보존 기간을 재정의할 수 없습니다. 항상 최소 168시간의 `RETAIN` 값을 지정하거나, 기본값을 사용하려면 `RETAIN`을 완전히 생략하십시오.

In [ ]:
%sql
VACUUM automotive_catalog.governance_lab.vehicle_telemetry RETAIN 168 HOURS;

### 작업 3.4 — 예측 최적화 활성화

`governance_lab` 스키마에서 **예측 최적화**를 활성화하여 향후 `VACUUM` 및 `OPTIMIZE` 유지보수 실행이 수동 일정 없이 자동으로 실행되도록 합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks의 Unity Catalog 스키마에서 예측 최적화를 활성화하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER SCHEMA ... ENABLE PREDICTIVE OPTIMIZATION`을 사용하십시오.

In [ ]:
%sql
ALTER SCHEMA automotive_catalog.governance_lab
ENABLE PREDICTIVE OPTIMIZATION;

## 연습 4: 프로그래매틱 방식으로 데이터 계보 쿼리

> 📋 **이 연습을 실행하기 전에**: 아직 수행하지 않았다면 여기서 일시 정지하고 실습 설정 지침의 **Catalog Explorer 계보 단계**를 따라 UI에서 계보를 시각적으로 탐색합니다. 그런 다음 여기로 돌아와 SQL을 사용하여 계보 데이터를 쿼리합니다.

Unity Catalog는 모든 읽기 및 쓰기 이벤트를 캡처하고 `system.access.table_lineage` 시스템 테이블에 저장합니다. 이를 통해 *"이번 주 자동차 카탈로그의 어느 테이블이 가장 많이 쿼리되었습니까?"*와 같은 질문에 프로그래매틱 방식으로 답할 수 있습니다.

### 작업 4.1 — 카탈로그에서 최근에 접근한 테이블 찾기

`system.access.table_lineage`를 쿼리하여 지난 7일 동안 `automotive_catalog`와 관련된 모든 **쓰기 이벤트**(원본 테이블)를 찾습니다. 테이블 이름과 고유 이벤트 수를 반환하며, 가장 활동적인 테이블순으로 정렬합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks에서 system.access.table_lineage를 쿼리하여 최근에 접근한 Unity Catalog 테이블을 찾으려면 어떻게 해야 합니까?"*

**힌트:** `source_table_full_name LIKE 'automotive_catalog%'`를 필터링하고 테이블 이름으로 그룹화합니다.

In [ ]:
%sql
SELECT
  source_table_full_name,
  COUNT(DISTINCT event_id) AS event_count
FROM system.access.table_lineage
WHERE
  event_date > CURRENT_DATE() - INTERVAL 7 DAYS
  AND source_table_full_name LIKE 'automotive_catalog%'
GROUP BY source_table_full_name
ORDER BY event_count DESC;

### 작업 4.2 — vehicle_telemetry의 히스토리 보기

`DESCRIBE HISTORY`를 사용하여 `vehicle_telemetry`의 전체 Delta Lake 작업 히스토리를 봅니다. 삭제가 발생한 버전(`DELETE` 작업)을 식별합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks SQL의 Delta Lake 테이블의 전체 버전 히스토리를 보려면 어떻게 해야 합니까?"*

In [ ]:
%sql
DESCRIBE HISTORY automotive_catalog.governance_lab.vehicle_telemetry;

## 연습 5: 감사 로깅

AutoSphere의 준수 팀은 지난 30일 동안 권한이 있는 사용자만 고객 데이터에 액세스했다는 것을 규제 기관에 증명해야 합니다. 팀은 또한 Unity Catalog 메타스토어의 권한이 수정될 때마다 경고를 받고 싶습니다.

`system.access.audit` 시스템 테이블을 사용하여 이러한 요구사항에 답합니다.

### 작업 5.1 — customer_registrations 테이블에 접근한 사람 찾기

`system.access.audit`를 쿼리하여 지난 30일 동안 `automotive_catalog.governance_lab.customer_registrations`에 접근한 모든 사용자를 나열합니다. 각 액세스의 작업 이름과 시간을 포함하며, 가장 최근 순서로 정렬합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks 감사 로그 시스템 테이블을 쿼리하여 특정 Unity Catalog 테이블에 접근한 사람을 찾으려면 어떻게 해야 합니까?"*

**힌트:** `request_params.full_name_arg` 및 `action_name IN ('getTable', 'createTable')`를 필터링합니다. 사용자 열에는 `user_identity.email`을 사용합니다.

In [ ]:
%sql
SELECT
  user_identity.email  AS user_email,
  action_name          AS access_type,
  event_time           AS access_time
FROM system.access.audit
WHERE
  request_params.full_name_arg = 'automotive_catalog.governance_lab.customer_registrations'
  AND action_name IN ('getTable', 'createTable', 'deleteTable')
  AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC;

### 작업 5.2 — 카탈로그 전체의 권한 변경 감지

`system.access.audit`를 쿼리하여 지난 30일 동안 `automotive_catalog`의 보안 가능 객체에 대한 모든 **권한 변경 이벤트**(`updatePermissions` 동작)를 나열합니다. 변경을 수행한 사람과 영향을 받은 객체를 포함합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"특정 Unity Catalog 카탈로그에 대한 Databricks 감사 로그에서 모든 권한 변경을 찾으려면 어떻게 해야 합니까?"*

**힌트:** `service_name = 'unityCatalog'`, `action_name = 'updatePermissions'`, `request_params.securable_full_name LIKE 'automotive_catalog%'`를 필터링합니다.

In [ ]:
%sql
SELECT
  event_time,
  user_identity.email                      AS changed_by,
  request_params.securable_type            AS object_type,
  request_params.securable_full_name       AS object_name,
  request_params.changes                   AS permission_changes
FROM system.access.audit
WHERE
  service_name = 'unityCatalog'
  AND action_name = 'updatePermissions'
  AND request_params.securable_full_name LIKE 'automotive_catalog%'
  AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC;

### 작업 5.3 — 사용자별 감사 활동 요약

지난 7일 동안 Unity Catalog 작업에 대한 사용자당 감사 이벤트 총 개수를 세는 요약 쿼리를 만듭니다. 이는 준수 팀에 플랫폼에서 가장 활동적이었던 사람을 빠르게 파악할 수 있게 합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks 감사 로그를 집계하여 Unity Catalog 활동에 대해 사용자당 이벤트를 계산하려면 어떻게 해야 합니까?"*

In [ ]:
%sql
SELECT
  user_identity.email  AS user_email,
  COUNT(*)             AS total_events
FROM system.access.audit
WHERE
  service_name = 'unityCatalog'
  AND event_date > CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY user_identity.email
ORDER BY total_events DESC;